In [ ]:
import cv2
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import decord
import json
import os
from tqdm.notebook import tqdm

In [ ]:
# Load video
video_path = r'/home/share/schaer2/idtracking_keypoint/input/20200505155636_20200505174320_0_converted_25fps.mp4'
# cap = cv2.VideoCapture(video_path)
vr = decord.VideoReader(video_path)
# Load JSON file
json_path = r'/home/share/schaer2/idtracking_keypoint/output/20200505155636_20200505174320_0_converted_small_mask_0-end.json'
output_dir = r'/home/share/schaer2/idtracking_keypoint/output'
os.makedirs(output_dir, exist_ok=True)
with open(json_path, 'r') as f:
    mask_data = json.load(f)


In [ ]:
width, height = vr[0].shape[1], vr[0].shape[0]
print(f"Width: {width}, Height: {height}")

In [ ]:
video_source_mask = r"/home/share/schaer2/idtracking_keypoint/input/20200505155636_20200505174320_0_converted_small.mp4"
cap = cv2.VideoCapture(video_source_mask)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"Width: {width}, Height: {height}")

In [ ]:
# Compute the scaling ratios between the mask source video and the vr video
cap = cv2.VideoCapture(video_source_mask)
src_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
src_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

vr_width = vr[0].shape[1]
vr_height = vr[0].shape[0]

ratio_x = vr_width / src_width
ratio_y = vr_height / src_height

print(f"Scale ratios - X: {ratio_x}, Y: {ratio_y}")

In [ ]:
def mask_to_bbox(mask):
    """Convert a binary mask to a bounding box."""
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if rows.any():
        y_min, y_max = np.where(rows)[0][[0, -1]]
        x_min, x_max = np.where(cols)[0][[0, -1]]
        return [int(x_min), int(y_min), int(x_max), int(y_max)]
    return [0, 0, 0, 0]

def rle_to_mask(rle) -> np.ndarray:
    """Compute a binary mask from an uncompressed RLE."""
    h, w = rle["size"]
    mask = np.empty(h * w, dtype=bool)
    idx = 0
    parity = False
    for count in rle["counts"]:
        mask[idx : idx + count] = parity
        idx += count
        parity ^= True
    mask = mask.reshape(w, h)
    return mask.transpose()  # Put in C order

def get_mask(frame_number, mask_data):
    """Retrieve the masks associated with a specific frame."""
    frames = mask_data.get('frames')
    if frame_number < len(frames):
        frame = frames[frame_number]
        points = frame.get('points', {})
        bboxs = frame.get('bboxs', {})
        masks =  frame.get('masks', [])
        return points, bboxs, masks
    return None

# Dictionary to store bounding boxes with image id as key
bbox_dict = {}
expected_ids = [0, 1, 2]
missing_data = []



RESIZED = True
for last_id in tqdm(range(0, len(vr))):
    _, _, masks = get_mask(last_id, mask_data)
    if masks:
        bbox_dict[str(last_id)] = {}
        for target_id, rle_mask in zip(masks.get('target_ids'), masks.get('rle_masks')):
            mask = rle_to_mask(rle_mask)
            # Resize mask to match vr frame size using the ratios
            if RESIZED:
                mask = cv2.resize(mask.astype(np.uint8), (vr_width, vr_height), interpolation=cv2.INTER_NEAREST)
            bbox = mask_to_bbox(mask)
            if bbox == [0, 0, 0, 0]:
                missing_data.append((last_id, target_id))
            else:
                # normalization 
                bbox = [bbox[0]/vr_width, bbox[1]/vr_height, bbox[2]/vr_width, bbox[3]/vr_height]
                # Save bounding box in dictionary
                bbox_dict[str(last_id)][target_id] = bbox
    else:
        for i in expected_ids:
            missing_data.append((last_id, i))
    

# Save bbox_dict as a JSON file
# output_json_path = os.path.join(output_dir, os.path.basename(video_path).split('.')[0] + '_bbox_source_size.json')
# with open(output_json_path, 'w') as outfile:
#     json.dump(bbox_dict, outfile)


In [ ]:
output_json_path = os.path.join(output_dir, os.path.basename(video_path).split('.')[0] + '_bbox.json')
with open(output_json_path, 'w') as outfile:
    json.dump(bbox_dict, outfile)